# PySpark Analysis

This notebook uses PySpark to analyze the cleaned Los Angeles Airbnb dataset.

In [1]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Set Java path for PySpark
JAVA_HOME = Path("/opt/homebrew/opt/openjdk@17")

os.environ["JAVA_HOME"] = str(JAVA_HOME)
os.environ["PATH"] = f"{JAVA_HOME / 'bin'}{os.pathsep}{os.environ['PATH']}"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Airbnb PySpark Analysis")
    .getOrCreate()
)

spark

In [4]:
DATA_PATH = "../data/processed/clean_listings.csv"

spark_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(DATA_PATH)
)

spark_df.head(5)

[Row(id=2708, price=67.22, log_price=4.2227377769905, host_is_superhost=1.0, neighbourhood_cleansed='Hollywood', latitude=34.09625, longitude=-118.34605, property_type='Private room in rental unit', room_type='Private room', accommodates=1, bathrooms=1.0, bedrooms=None, beds=1.0, minimum_nights=30.0, maximum_nights=1125.0, availability_365=301, number_of_reviews=47, review_scores_rating=4.87, review_scores_cleanliness=4.94, review_scores_location=4.96, review_scores_value=4.87, amenities_count=77),
 Row(id=2732, price=213.56, log_price=5.368589419533368, host_is_superhost=0.0, neighbourhood_cleansed='Santa Monica', latitude=34.0044, longitude=-118.48095, property_type='Private room in rental unit', room_type='Private room', accommodates=1, bathrooms=1.0, bedrooms=None, beds=1.0, minimum_nights=1.0, maximum_nights=27.0, availability_365=350, number_of_reviews=24, review_scores_rating=4.41, review_scores_cleanliness=4.58, review_scores_location=4.91, review_scores_value=4.22, amenities_c

## Check shape and columns

In [5]:
num_rows = spark_df.count()
num_cols = len(spark_df.columns)

print("Rows:", num_rows)
print("Columns:", num_cols)

Rows: 37422
Columns: 22


In [6]:
spark_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- log_price: double (nullable = true)
 |-- host_is_superhost: double (nullable = true)
 |-- neighbourhood_cleansed: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- accommodates: integer (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- beds: double (nullable = true)
 |-- minimum_nights: double (nullable = true)
 |-- maximum_nights: double (nullable = true)
 |-- availability_365: integer (nullable = true)
 |-- number_of_reviews: integer (nullable = true)
 |-- review_scores_rating: double (nullable = true)
 |-- review_scores_cleanliness: double (nullable = true)
 |-- review_scores_location: double (nullable = true)
 |-- review_scores_value: double (nullable = true)
 |-- amenities_count: integer (nullable = true)


## Basic summary statistics

In [7]:
spark_df.select(
    F.round(F.avg("price"), 2).alias("avg_price"),
    F.round(F.expr("percentile_approx(price, 0.5)"), 2).alias("median_price"),
    F.round(F.min("price"), 2).alias("min_price"),
    F.round(F.max("price"), 2).alias("max_price"),
    F.count("*").alias("num_listings")
).show()

+---------+------------+---------+---------+------------+
|avg_price|median_price|min_price|max_price|num_listings|
+---------+------------+---------+---------+------------+
|   337.97|      223.57|    34.75|   2915.0|       37422|
+---------+------------+---------+---------+------------+



## Average price by booking type

In [8]:
room_type_summary_spark = (
    spark_df
    .groupBy("room_type")
    .agg(
        F.count("*").alias("num_listings"),
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.round(F.expr("percentile_approx(price, 0.5)"), 2).alias("median_price")
    )
    .orderBy(F.desc("avg_price"))
)

room_type_summary_spark.show(truncate=False)

+---------------+------------+---------+------------+
|room_type      |num_listings|avg_price|median_price|
+---------------+------------+---------+------------+
|Hotel room     |396         |485.71   |314.28      |
|Entire home/apt|27983       |402.22   |279.59      |
|Private room   |8898        |132.83   |86.0        |
|Shared room    |145         |122.37   |50.88       |
+---------------+------------+---------+------------+



## Top neighborhoods by average price

In [ ]:
neighborhood_summary_spark = (
    spark_df
    .groupBy("neighbourhood_cleansed")
    .agg(
        F.count("*").alias("num_listings"),
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.round(F.expr("percentile_approx(price, 0.5)"), 2).alias("median_price")
    )
    .filter(F.col("num_listings") >= 30)
    .orderBy(F.desc("avg_price"))
)

neighborhood_summary_spark.show(15, truncate=False)

+-------------------------------------+------------+---------+------------+
|neighbourhood_cleansed               |num_listings|avg_price|median_price|
+-------------------------------------+------------+---------+------------+
|Beverly Crest                        |160         |1194.74  |1077.65     |
|Malibu                               |302         |1039.39  |850.93      |
|Hollywood Hills West                 |545         |987.65   |853.33      |
|Bel-Air                              |54          |985.0    |831.14      |
|Avalon                               |259         |914.97   |848.0       |
|Unincorporated Santa Monica Mountains|161         |833.34   |616.33      |
|Manhattan Beach                      |410         |735.55   |579.5       |
|Pacific Palisades                    |93          |625.1    |413.11      |
|Veterans Administration              |51          |624.22   |548.25      |
|Fairfax                              |216         |567.78   |254.43      |
|Tarzana    

## Rank neighborhoods

In [28]:
from pyspark.sql import Window

neighborhood_room_summary = (
    spark_df
    .groupBy("room_type", "neighbourhood_cleansed")
    .agg(
        F.count("*").alias("num_listings"),
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.round(F.expr("percentile_approx(price, 0.5)"), 2).alias("median_price")
    )
    .filter(F.col("num_listings") >= 30)
    .orderBy(F.desc("avg_price"))
)

price_rank_window = Window.partitionBy("room_type").orderBy(F.desc("avg_price"))

ranked_neighborhoods = (
    neighborhood_room_summary
    .withColumn("price_rank", F.rank().over(price_rank_window))
    .select(
        "room_type",
        "price_rank",
        "neighbourhood_cleansed",
        "num_listings",
        "avg_price",
        "median_price"
    )
    .orderBy("room_type", "price_rank")
)

ranked_neighborhoods.show(30, truncate=False)

+---------------+----------+-------------------------------------+------------+---------+------------+
|room_type      |price_rank|neighbourhood_cleansed               |num_listings|avg_price|median_price|
+---------------+----------+-------------------------------------+------------+---------+------------+
|Entire home/apt|1         |Beverly Crest                        |146         |1279.02  |1225.0      |
|Entire home/apt|2         |Bel-Air                              |42          |1201.98  |1005.0      |
|Entire home/apt|3         |Malibu                               |284         |1063.35  |870.0       |
|Entire home/apt|4         |Hollywood Hills West                 |499         |1053.54  |947.27      |
|Entire home/apt|5         |Avalon                               |241         |938.12   |869.5       |
|Entire home/apt|6         |Unincorporated Santa Monica Mountains|153         |862.84   |670.0       |
|Entire home/apt|7         |Manhattan Beach                      |377    

## Average price by guest capacity

In [12]:
guest_capacity_summary = (
    spark_df
    .groupBy("accommodates")
    .agg(
        F.count("*").alias("num_listings"),
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.round(F.expr("percentile_approx(price, 0.5)"), 2).alias("median_price")
    )
    .filter(F.col("num_listings") >= 30)
    .orderBy(F.desc("accommodates"))
)

guest_capacity_summary.show(20, truncate=False)

+------------+------------+---------+------------+
|accommodates|num_listings|avg_price|median_price|
+------------+------------+---------+------------+
|16          |286         |1006.34  |829.0       |
|15          |49          |1076.85  |834.0       |
|14          |189         |917.4    |708.5       |
|13          |55          |692.22   |596.0       |
|12          |473         |902.58   |708.0       |
|11          |127         |699.61   |501.0       |
|10          |1168        |934.83   |740.88      |
|9           |347         |578.55   |464.01      |
|8           |2648        |702.19   |525.0       |
|7           |881         |497.07   |407.34      |
|6           |4491        |502.08   |387.5       |
|5           |1906        |372.63   |306.52      |
|4           |6625        |305.29   |246.0       |
|3           |2797        |223.8    |183.1       |
|2           |12389       |168.45   |127.14      |
|1           |2991        |96.67    |71.93       |
+------------+------------+----

In [29]:
REPORTS_DIR = Path("../reports")

room_type_summary_spark.toPandas().to_csv(
    REPORTS_DIR / "pyspark_room_type_summary.csv",
    index=False
)

ranked_neighborhoods.toPandas().to_csv(
    REPORTS_DIR / "pyspark_ranked_neighborhoods.csv",
    index=False
)

guest_capacity_summary.toPandas().to_csv(
    REPORTS_DIR / "pyspark_guest_capacity_summary.csv",
    index=False
)

print("Saved PySpark summary files")

Saved PySpark summary files


## Takeaways

The cleaned Airbnb dataset contains 37,422 listings and 22 columns. The average nightly price is about $338, while the median price is about $224, showing that prices are right-skewed as higher priced listings pull the average upward.

Room type has a clear relationship with price. Hotel rooms and entire homes/apartments have the highest average prices, while private rooms and shared rooms are much cheaper on average.

Location also plays a major role in pricing. Neighborhoods such as Beverly Crest, Malibu, Hollywood Hills West, Bel-Air, and Avalon had some of the highest average nightly prices.

Guest capacity seems to also play an important factor. Larger listings generally have higher average and median prices, especially listings that accommodate 10 or more guests.

Overall, the analysis supports the main modeling results that Airbnb prices are strongly influenced by location, booking type, and listing size.